In [1]:
import cv2 
print(cv2.__version__)


4.13.0


In [2]:
import mediapipe


In [3]:
import cv2
import mediapipe as mp

print("MediaPipe version:", mp.__version__)
print("Solutions available:", dir(mp.solutions))

MediaPipe version: 0.10.9
Solutions available: ['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'download_utils', 'drawing_styles', 'drawing_utils', 'face_detection', 'face_mesh', 'face_mesh_connections', 'hands', 'hands_connections', 'holistic', 'mediapipe', 'objectron', 'pose', 'pose_connections', 'selfie_segmentation']


In [4]:
import pyautogui


In [5]:
import mediapipe as mp
print(mp.__file__)

C:\Users\user\anaconda3\envs\opencv\Lib\site-packages\mediapipe\__init__.py


In [6]:
import mediapipe as mp
print(mp.__version__)
print(mp.solutions.hands)

0.10.9
<module 'mediapipe.python.solutions.hands' from 'C:\\Users\\user\\anaconda3\\envs\\opencv\\Lib\\site-packages\\mediapipe\\python\\solutions\\hands.py'>


In [ ]:
import cv2
import mediapipe as mp

mp_hands = mp.solutions.hands
hands = mp_hands.Hands()
mp_draw = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    if result.multi_hand_landmarks:
        for hand in result.multi_hand_landmarks:
            mp_draw.draw_landmarks(frame, hand, mp_hands.HAND_CONNECTIONS)

    cv2.imshow("Hand Tracking", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2
import mediapipe as mp
import pyautogui
import numpy as np

# Disable PyAutoGUI fail-safe (important)
pyautogui.FAILSAFE = False

# Screen size
screen_width, screen_height = pyautogui.size()

# MediaPipe setup
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)
mp_draw = mp.solutions.drawing_utils

# Camera
cap = cv2.VideoCapture(0)

while True:
    success, frame = cap.read()
    if not success:
        break

    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb)

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:

            # Index finger tip (id = 8)
            index_finger = hand_landmarks.landmark[8]
            x = int(index_finger.x * w)
            y = int(index_finger.y * h)

            # Convert camera coords to screen coords
            screen_x = np.interp(x, (0, w), (0, screen_width))
            screen_y = np.interp(y, (0, h), (0, screen_height))

            # Move mouse
            pyautogui.moveTo(screen_x, screen_y)

            # Draw landmarks
            mp_draw.draw_landmarks(
                frame,
                hand_landmarks,
                mp_hands.HAND_CONNECTIONS
            )

            # Thumb tip (id = 4) for click
            thumb = hand_landmarks.landmark[4]
            thumb_x = int(thumb.x * w)
            thumb_y = int(thumb.y * h)

            # Distance between thumb & index
            distance = np.hypot(thumb_x - x, thumb_y - y)

            # Click when fingers are close
            if distance < 30:
                pyautogui.click()
                pyautogui.sleep(0.2)

    cv2.imshow("Hand Mouse Control", frame)

    # Press ESC to exit
    if cv2.waitKey(100) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()